In [80]:
import os
import re
import rasterio
import numpy as np

from datetime import datetime
# ficheros landsat tienen nombre de fecha, region del espectro que tienen (blue, green etc) y el numero de banda
class Landsat:
    """
    Clase para trabajar con imágenes Landsat.

    Esta clase permite organizar las bandas espectrales de una escena Landsat,
    leer metadatos y calcular índices espectrales como NDVI y MNDWI.

    Parameters
    ----------
    path : str
        Ruta al directorio que contiene los archivos de la escena Landsat.

    Attributes
    ----------
    path : str
        Ruta original de la escena.
    ori : str
        Directorio padre inmediato.
    raiz : str
        Directorio raíz del proyecto.
    dat : str
        Carpeta para datos.
    pro : str
        Carpeta para resultados procesados.
    escena : str
        Nombre de la escena.
    pro_escena : str
        Carpeta de salida específica de la escena.
    blue, green, red, nir, swir1, swir2 : str
        Rutas a las bandas espectrales.
    fmask : str
        Ruta a la máscara de nubes (si existe).
    hillshade : str
        Ruta al hillshade (si existe).
    mtl : str
        Ruta al archivo de metadatos.
    dmtl : dict
        Diccionario con metadatos del archivo MTL.
    """

    def __init__(self, path):
        """
        Constructor.
        
        Args:
            path (str): path de mi Landsat
            
        """
        self.path = path # self para decir que trabajamos en uno en particular (en path, en raza etc..)
        self.ori = os.path.split(self.path)[0]
        self.raiz = os.path.split(self.ori)[0]
        self.dat = os.path.join(self.raiz, 'dat')
        os.makedirs(self.dat, exist_ok = True)
        self.pro = os.path.join(self.raiz, 'pro')
        os.makedirs(self.pro, exist_ok = True)
        self.escena = os.path.split(self.path)[1]
        self.pro_escena = os.path.join(self.pro, self.escena)
        os.makedirs(self.pro_escena, exist_ok = True)

        for i in os.listdir(self.path):
            if i.endswith('.tif'): # queremos solamente los archivos que terminan con .tif
                if len(i.split('_')) > 3:
                    espectro = i.split('_')[-2]  # queremos partir el nombre de cada archivo por _ 
                    if espectro == 'blue':
                        self.blue = os.path.join(self.path, i)
                    elif espectro == 'green':
                        self.green = os.path.join(self.path, i)
                    elif espectro == 'red':
                        self.red = os.path.join(self.path, i)
                    elif espectro == 'nir':
                        self.nir = os.path.join(self.path, i)
                    elif espectro == 'swir1':
                        self.swir1 = os.path.join(self.path, i)
                    elif espectro == 'swir2':
                        self.swir2 = os.path.join(self.path, i)
        for i in os.listdir(self.path):
                #print(banda)
                if re.search('tif$', i): # Hace lo mismo que el endswith, igual que el regex de r
                    if 'fmask' in i:
                        self.fmask = os.path.join(self.path, i)
                    elif 'hillshade' in i:
                        self.hillshade= os.path.join(self.path, i)
        for i in os.listdir(self.path):
            if re.search('txt$', i):
                self.mtl = os.path.join(self.path, i)
        self.dmtl= {}
        with open(self.mtl, 'r') as f: #abrir el archivo
            for line in f.readlines(): #que recorra las lineas
                if '=' in line: #si hay un igual en la linea
                    l= line.split('=') # partimos la linea por el '='
                    self.dmtl[l[0].strip()]= l[1].strip() # asignamos la calve y el valor dentro del diccionario y el strip le quita el ultimo caracter
            
        print(f'Imagen importada de {path}')

    def ndvi(self):
        """
        Calcula el índice NDVI (Normalized Difference Vegetation Index).

        El NDVI se define como:

            NDVI = (NIR - RED) / (NIR + RED)

        donde:
        - NIR es la banda infrarroja cercana
        - RED es la banda roja

        Returns
        -------
        None
            El resultado se guarda como archivo raster en disco.

        Notes
        -----
        - El resultado se guarda en formato GeoTIFF.
        - Los valores nodata se establecen en -9999.
        - El tipo de dato de salida es float32.
        """
        print('Calculando ndvi')
        with rasterio.open(self.nir) as nir:
            NIR = nir.read()
        with rasterio.open(self.red) as red:
            RED = red.read()
        num = NIR.astype(float) - RED.astype(float)
        den = NIR + RED
        #ndvi = num/den
        ndvi = np.true_divide(num, den) # como un array

        profile = nir.meta
        profile.update(nodata=-9999)
        profile.update(dtype=rasterio.float32)
        #meta.update({'dtype': 'int16', 'nodata': None})
        
        output_file = os.path.join(self.pro_escena, self.escena + '_ndvi.tif')
        with rasterio.open(output_file, 'w', **profile) as dst:
            dst.write(ndvi.astype(rasterio.float32))
        print('Ndvi calculado')

    def mndwi(self):
        """ 
        The Modified Normalized Difference Water Index (MNDWI) uses green and SWIR bands for the enhancement of open water features. It also diminishes built-up area features that are often correlated with open water in other indices.

        MNDWI = (Green - SWIR) / (Green + SWIR)
    
        Green = pixel values from the green band
        SWIR = pixel values from the short-wave infrared band
    
        Reference: Xu, H. "Modification of Normalised Difference Water Index (NDWI) to Enhance Open Water Features in Remotely Sensed Imagery." International Journal of Remote Sensing 27, No. 14 (2006): 3025-3033." (ESRI, 2018)
    
        Sources
        "Indices gallery". ArcGIS Pro, ESRI. 2018. 
        http://pro.arcgis.com/en/pro-app/help/data/imagery/indices-gallery.htm.
        Accessed February 1, 2019.

        Calcula el índice MNDWI (Modified Normalized Difference Water Index).

        El MNDWI se define como:

            MNDWI = (GREEN - SWIR1) / (GREEN + SWIR1)

        donde:
        - GREEN es la banda verde
        - SWIR1 es la banda infrarroja de onda corta

        Returns
        -------
        None
            El resultado se guarda como archivo raster en disco.

        Notes
        -----
        - Mejora la detección de cuerpos de agua.
        - Reduce la interferencia de áreas urbanas.
        - El resultado se guarda como GeoTIFF con tipo float32.
        """
        print('Calculando mndwi')
        with rasterio.open(self.green) as green:
            GREEN = green.read()
        with rasterio.open(self.swir1) as swir:
            SWIR = swir.read()
        num = GREEN.astype(float) - SWIR.astype(float)
        den = GREEN + SWIR
        #ndvi = num/den
        mndwi = np.true_divide(num, den) # como un array

        profile = green.meta
        profile.update(nodata=-9999)
        profile.update(dtype=rasterio.float32)
        #meta.update({'dtype': 'int16', 'nodata': None})
        
        output_file = os.path.join(self.pro_escena, self.escena + '_mndwi.tif')
        with rasterio.open(output_file, 'w', **profile) as dst:
            dst.write(mndwi.astype(rasterio.float32))
        print('Mndwi calculado')

    def run(self):
        """
        Ejecuta el cálculo de índices espectrales.

        Este método llama secuencialmente a:
        - ndvi()
        - mndwi()

        Returns
        -------
        None
            Los resultados se guardan en disco.
        """
        self.ndvi()
        self.mndwi()

In [81]:
base = r'C:\Users\gaiam\Geopython_2026-main\dia_5\ori'
for i in os.listdir(base):
    landsat = Landsat(os.path.join(base, i))
    landsat.run()

Imagen importada de C:\Users\gaiam\Geopython_2026-main\dia_5\ori\20250319l8oli202_34
Calculando ndvi


C:\Users\gaiam\AppData\Local\Temp\ipykernel_31460\2664540884.py:79: RuntimeWarning: invalid value encountered in divide
  ndvi = np.true_divide(num, den) # como un array


Ndvi calculado
Calculando mndwi
Mndwi calculado
Imagen importada de C:\Users\gaiam\Geopython_2026-main\dia_5\ori\20250327l9oli202_34
Calculando ndvi
Ndvi calculado
Calculando mndwi
Mndwi calculado


In [79]:
landsat.run() # con el bucle de antes he creado este atributo así que cada vez que necesito una banda especifica me coge la ruta especifica del archivo 

Calculando ndvi


C:\Users\gaiam\AppData\Local\Temp\ipykernel_31460\2664540884.py:79: RuntimeWarning: invalid value encountered in divide
  ndvi = np.true_divide(num, den) # como un array


Ndvi calculado
Calculando mndwi
Mndwi calculado


In [72]:
landsat.ndvi()

C:\Users\gaiam\AppData\Local\Temp\ipykernel_31460\391088244.py:78: RuntimeWarning: invalid value encountered in divide
  ndvi = np.true_divide(num, den) # como un array


In [75]:
landsat.mndwi()